In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

In [3]:
project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").is_file():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook from inside the project repository.")
    project_root = project_root.parent

In [4]:
db_path = project_root / "1_SQL" / "analytics.db"
conn = sqlite3.connect(db_path)
print(f"Connected to database: {db_path}")

Connected to database: D:\Data Analysis\_My_Projects\dataanalytics_challenge\1_SQL\analytics.db


# Store and Format Performance

In [5]:
## Check the stores profit and rent 
## Check staffing cost in regards to the store region
## Shop sqft with revenue 
## Peak hours and dead hours different from transport-hub and high-street and/or region
## Does avg_time_spent_by_customers gaining more revenue 
## how much is footfall, in high-street are there other same coffee chains
## Whats the staff salary between the two store format 

In [6]:
# Which store format (High-Street, Retail-Park, Transport-Hub) 
# delivers the highest average `net_profit_margin_pct`?

print("Monthly")
query = """
SELECT
    s.store_format,
    COUNT(DISTINCT m.store_id)                                         AS stores,
    COUNT(*)                                                           AS store_months,
    ROUND(AVG(m.net_profit_margin_pct) * 100, 2)                       AS avg_monthly_margin_pct,
    ROUND(100.0 * SUM(m.net_profit_gbp) / SUM(m.total_monthly_revenue_gbp), 2) AS overall_margin_pct,
    ROUND(SUM(m.net_profit_gbp), 0)                                    AS total_net_profit_gbp
FROM fact_store_monthly_profitability AS m
JOIN dim_store AS s
    ON s.store_id = m.store_id
GROUP BY s.store_format
ORDER BY avg_monthly_margin_pct DESC;
"""
result = pd.read_sql_query(query, conn)
result


## Insights
# Transport-Hub has the highest average margin among the formats present, ahead by 8.18 percentage points.


Monthly


,store_format,stores,store_months,avg_monthly_margin_pct,overall_margin_pct,total_net_profit_gbp
0,Transport-Hub,2,48,22.53,25.14,801111.0
1,High-Street,58,1392,14.34,18.59,16078393.0


In [7]:
# Which UK regions show the highest concentration of `profitability_status` = Loss-Making or Structural-Loss?
# highest concentration means percentage of stores affected

print("Monthly")
query = """
SELECT
    s.region,
    COUNT(DISTINCT m.store_id)                                         AS stores,
    COUNT(*)                                                           AS store_months,
    SUM(CASE WHEN m.profitability_status IN ('Loss-Making', 'Structural-Loss')
             THEN 1 ELSE 0 END)                                        AS label_loss_months,
    ROUND(100.0 * SUM(CASE WHEN m.profitability_status IN ('Loss-Making', 'Structural-Loss')
                           THEN 1 ELSE 0 END) / COUNT(*), 1)           AS label_loss_pct,
    SUM(CASE WHEN m.net_profit_margin_pct < -0.02 THEN 1 ELSE 0 END)   AS calc_loss_months,
    ROUND(100.0 * SUM(CASE WHEN m.net_profit_margin_pct < -0.02
                           THEN 1 ELSE 0 END) / COUNT(*), 1)           AS calc_loss_pct
FROM fact_store_monthly_profitability AS m
JOIN dim_store AS s
    ON s.store_id = m.store_id
GROUP BY s.region
ORDER BY calc_loss_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

## Insights
# - 

Monthly


,region,stores,store_months,label_loss_months,label_loss_pct,calc_loss_months,calc_loss_pct
0,London,7,168,43,25.6,49,29.2
1,South-West,5,120,18,15.0,23,19.2
2,North-West,5,120,19,15.8,21,17.5
3,East-Midlands,5,120,17,14.2,20,16.7
4,North-East,4,96,8,8.3,13,13.5
5,West-Midlands,3,72,6,8.3,9,12.5
6,Scotland,4,96,8,8.3,10,10.4
7,South-East,6,144,12,8.3,14,9.7
8,East-of-England,6,144,12,8.3,13,9.0
9,Northern-Ireland,3,72,5,6.9,6,8.3


In [8]:
# Do stores with higher `monthly_rent_gbp` necessarily generate higher 
# `total_monthly_revenue_gbp`, or does rent erode profit disproportionately in some formats?

print("Monthly")
query = """
SELECT
    s.store_format,
    m.rent_band,
    COUNT(DISTINCT m.store_id)                                         AS stores,
    COUNT(*)                                                           AS store_months,
    ROUND(AVG(m.monthly_rent_gbp), 0)                                  AS avg_monthly_rent_gbp,
    ROUND(AVG(m.total_monthly_revenue_gbp), 0)                         AS avg_monthly_revenue_gbp,
    ROUND(100.0 * SUM(m.monthly_rent_gbp) / SUM(m.total_monthly_revenue_gbp), 2) AS rent_pct_of_revenue,
    ROUND(100.0 * SUM(m.net_profit_gbp)   / SUM(m.total_monthly_revenue_gbp), 2) AS margin_pct
FROM fact_store_monthly_profitability AS m
JOIN dim_store AS s
    ON s.store_id = m.store_id
GROUP BY s.store_format, m.rent_band
ORDER BY s.store_format,
         CASE m.rent_band WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2
                          WHEN 'High' THEN 3 WHEN 'Premium' THEN 4 END;

"""

result = pd.read_sql_query(query, conn)
result

# INSIGHT 1
# - Higher rent share, lower profit margin: As the rent share increases across these groups, the profit margin decreases. This suggests rent pressure may be associated with lower profitability.
# For High-Street stores:
# - Low rent band: rent consumes 6.60% of revenue; margin is 21.41%.
# - High rent band: rent consumes 19.51% of revenue; margin is 10.58%.

# INSIGHT 2
# - Transport-Hub stores have higher margins in this sample
# * Low rent band: 26.58% margin.
# * Medium rent band: 23.85% margin.



Monthly


,store_format,rent_band,stores,store_months,avg_monthly_rent_gbp,avg_monthly_revenue_gbp,rent_pct_of_revenue,margin_pct
0,High-Street,Low,29,551,4042.0,61218.0,6.60,21.41
1,High-Street,Medium,35,690,7148.0,62653.0,11.41,18.15
2,High-Street,High,11,151,12321.0,63160.0,19.51,10.58
3,Transport-Hub,Low,1,22,4619.0,68402.0,6.75,26.58
4,Transport-Hub,Medium,2,26,6950.0,64680.0,10.75,23.85


In [9]:
print("Monthly")
query = """
SELECT
    s.store_id,
    s.store_name,
    s.store_format,
    ROUND(AVG(m.monthly_rent_gbp), 0)                                  AS avg_monthly_rent_gbp,
    ROUND(AVG(m.total_monthly_revenue_gbp), 0)                         AS avg_monthly_revenue_gbp,
    ROUND(100.0 * SUM(m.monthly_rent_gbp) / SUM(m.total_monthly_revenue_gbp), 2) AS rent_pct_of_revenue,
    ROUND(100.0 * SUM(m.net_profit_gbp)   / SUM(m.total_monthly_revenue_gbp), 2) AS margin_pct
FROM fact_store_monthly_profitability AS m
JOIN dim_store AS s
    ON s.store_id = m.store_id
GROUP BY s.store_id, s.store_name, s.store_format
ORDER BY avg_monthly_rent_gbp DESC;

"""

result = pd.read_sql_query(query, conn)
result

Monthly


,store_id,store_name,store_format,avg_monthly_rent_gbp,avg_monthly_revenue_gbp,rent_pct_of_revenue,margin_pct
0,15,Sunrise Coffee West London Quarter,High-Street,14303.0,64860.0,22.05,20.54
1,5,Sunrise Coffee Canary Wharf Quarter,High-Street,12452.0,63008.0,19.76,8.69
2,43,Sunrise Coffee North London Yard,High-Street,12293.0,60149.0,20.44,5.14
3,53,Sunrise Coffee Canary Wharf Express,High-Street,12262.0,56297.0,21.78,3.17
4,46,Sunrise Coffee East London Lounge,High-Street,12253.0,71382.0,17.17,9.61
5,39,Sunrise Coffee Central London House,High-Street,10976.0,62415.0,17.59,13.46
6,47,Sunrise Coffee Exeter Express,High-Street,9391.0,61392.0,15.30,10.07
7,24,Sunrise Coffee Cheltenham Quarter,High-Street,9290.0,67466.0,13.77,17.28
8,14,Sunrise Coffee Oxford Lounge,High-Street,9153.0,61779.0,14.82,22.77
9,20,Sunrise Coffee Guildford Hub,High-Street,9105.0,56724.0,16.05,13.15


In [10]:
# Which individual stores improved their `net_profit_gbp` most from 2023 to 2024, 
# and which have deteriorated?

print("Monthly")
query = """
WITH yearly AS (
    SELECT
        store_id,
        SUM(CASE WHEN month_id LIKE '2023-%' THEN net_profit_gbp END)  AS profit_2023,
        SUM(CASE WHEN month_id LIKE '2024-%' THEN net_profit_gbp END)  AS profit_2024
    FROM fact_store_monthly_profitability
    GROUP BY store_id
)
SELECT
    s.store_id,
    s.store_name,
    s.store_format,
    s.region,
    ROUND(y.profit_2023, 0)                                            AS profit_2023_gbp,
    ROUND(y.profit_2024, 0)                                            AS profit_2024_gbp,
    ROUND(y.profit_2024 - y.profit_2023, 0)                            AS change_gbp,
    ROUND(100.0 * (y.profit_2024 - y.profit_2023)
          / NULLIF(ABS(y.profit_2023), 0), 1)                          AS change_pct,
    CASE WHEN y.profit_2024 > y.profit_2023 THEN 'Improved' ELSE 'Deteriorated' END AS direction
FROM yearly AS y
JOIN dim_store AS s
    ON s.store_id = y.store_id
ORDER BY change_gbp DESC;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,store_id,store_name,store_format,region,profit_2023_gbp,profit_2024_gbp,change_gbp,change_pct,direction
0,42,Sunrise Coffee Belfast Yard,High-Street,Northern-Ireland,150444.0,267234.0,116790.0,77.6,Improved
1,45,Sunrise Coffee Liverpool Collective,High-Street,North-West,80690.0,197187.0,116497.0,144.4,Improved
2,24,Sunrise Coffee Cheltenham Quarter,High-Street,South-West,94810.0,185034.0,90224.0,95.2,Improved
3,37,Sunrise Coffee Ipswich House,High-Street,East-of-England,98355.0,176371.0,78016.0,79.3,Improved
4,48,Sunrise Coffee Reading Hub,High-Street,South-East,148060.0,222801.0,74741.0,50.5,Improved
5,25,Sunrise Coffee Glasgow Express,High-Street,Scotland,109759.0,170855.0,61096.0,55.7,Improved
6,44,Sunrise Coffee Newcastle Collective,High-Street,North-East,73873.0,129401.0,55528.0,75.2,Improved
7,12,Sunrise Coffee Cardiff Corner,High-Street,Wales,165531.0,218452.0,52921.0,32.0,Improved
8,15,Sunrise Coffee West London Quarter,High-Street,London,133736.0,186007.0,52271.0,39.1,Improved
9,4,Sunrise Coffee Chester Brew Bar,High-Street,North-West,51943.0,99296.0,47352.0,91.2,Improved


# Loyalty Programme

In [11]:
# Is there a positive correlation between `loyalty_adoption_rate` and `loyalty_member_revenue_pct` at a store level?

print("Monthly")
query = """
SELECT 
    fs_mon.loyalty_member_revenue_pct AS loyalty_member_revenue_pct,
    str.loyalty_adoption_rate AS loyalty_adoption_rate
FROM fact_store_monthly_profitability AS fs_mon
LEFT JOIN dim_store AS str 
    ON fs_mon.store_id = str.store_id
"""

result = pd.read_sql_query(query, conn)

correlation = result['loyalty_member_revenue_pct'].corr(result['loyalty_adoption_rate'])
print("Correlation between  `loyalty_adoption_rate` and `loyalty_member_revenue_pct` is", correlation)

# Insights
## Strong positive correlation

Monthly
Correlation between  `loyalty_adoption_rate` and `loyalty_member_revenue_pct` is 0.9584402350078945


In [ ]:
# Do loyalty members spend more per transaction than non-members, 
# based on `avg_transaction_value_gbp` relative to `loyalty_transactions`?

print("Monthly")
query = """
WITH d AS (
    SELECT
        1.0 * loyalty_transactions / total_transactions                AS loyalty_txn_share,
        total_revenue_gbp,
        total_transactions
    FROM fact_daily_sales
)
SELECT
    CASE
        WHEN loyalty_txn_share < 0.20 THEN '1) under 20%'
        WHEN loyalty_txn_share < 0.40 THEN '2) 20-40%'
        WHEN loyalty_txn_share < 0.60 THEN '3) 40-60%'
        ELSE                               '4) 60% or more'
    END                                                                AS loyalty_share_band,
    COUNT(*)                                                           AS day_records,
    ROUND(100.0 * AVG(loyalty_txn_share), 1)                           AS avg_loyalty_txn_share_pct,
    ROUND(SUM(total_revenue_gbp) / SUM(total_transactions), 2)         AS avg_transaction_value_gbp
FROM d
GROUP BY loyalty_share_band
ORDER BY loyalty_share_band;
"""

result = pd.read_sql_query(query, conn)
result

# Insights
# 

Monthly


,loyalty_share_band,day_records,avg_loyalty_txn_share_pct,avg_transaction_value_gbp
0,1) under 20%,2943,10.9,7.30
1,2) 20-40%,3446,31.2,7.25
2,3) 40-60%,3520,48.4,7.30
3,4) 60% or more,91,61.6,7.09


In [13]:
print("Monthly")
query = """
WITH txn AS (
    SELECT store_id,
           1.0 * SUM(loyalty_transactions) / SUM(total_transactions)  AS member_txn_share
    FROM fact_daily_sales
    GROUP BY store_id
),
rev AS (
    SELECT store_id,
           SUM(loyalty_member_spend_gbp) / SUM(total_monthly_revenue_gbp) AS member_rev_share
    FROM fact_store_monthly_profitability
    GROUP BY store_id
)
SELECT
    s.store_id,
    s.store_name,
    s.store_format,
    ROUND(100.0 * t.member_txn_share, 1)                               AS member_txn_share_pct,
    ROUND(100.0 * r.member_rev_share, 1)                               AS member_rev_share_pct,
    ROUND(r.member_rev_share / t.member_txn_share, 2)                  AS spend_index
FROM dim_store AS s
JOIN txn AS t ON t.store_id = s.store_id
JOIN rev AS r ON r.store_id = s.store_id
ORDER BY spend_index DESC;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,store_id,store_name,store_format,member_txn_share_pct,member_rev_share_pct,spend_index
0,36,Sunrise Coffee Leicester Express,High-Street,7.4,9.2,1.23
1,39,Sunrise Coffee Central London House,High-Street,11.9,14.5,1.22
2,28,Sunrise Coffee Hull Hub,High-Street,10.1,11.9,1.18
3,34,Sunrise Coffee Brighton Express,High-Street,7.5,8.7,1.16
4,3,Sunrise Coffee Newport Yard,High-Street,39.3,43.6,1.11
5,33,Sunrise Coffee Aberdeen Express,High-Street,13.5,15.0,1.11
6,23,Sunrise Coffee Edinburgh Corner,High-Street,12.8,14.1,1.10
7,10,Sunrise Coffee Leeds Roastery,High-Street,18.7,20.1,1.08
8,21,Sunrise Coffee Birmingham Roastery,High-Street,51.8,56.0,1.08
9,24,Sunrise Coffee Cheltenham Quarter,High-Street,55.0,59.7,1.08


In [14]:
# What is the net value of the loyalty programme per store: loyalty member revenue contribution minus `loyalty_redemption_cost_gbp`?

print("Monthly")
query = """
SELECT
    s.store_id,
    s.store_name,
    s.store_format,
    s.region,
    ROUND(SUM(m.loyalty_member_spend_gbp), 0)                          AS member_revenue_gbp,
    ROUND(SUM(m.loyalty_redemption_cost_gbp), 0)                       AS redemption_cost_gbp,
    ROUND(SUM(m.loyalty_member_spend_gbp) - SUM(m.loyalty_redemption_cost_gbp), 0) AS net_value_gbp,
    ROUND(SUM(m.loyalty_member_spend_gbp
              * (1 - m.total_monthly_cogs_gbp / m.total_monthly_revenue_gbp))
          - SUM(m.loyalty_redemption_cost_gbp), 0)                     AS member_gp_after_cost_gbp,
    ROUND(100.0 * SUM(m.loyalty_redemption_cost_gbp)
          / SUM(m.loyalty_member_spend_gbp), 2)                        AS cost_pct_of_member_revenue
FROM fact_store_monthly_profitability AS m
JOIN dim_store AS s
    ON s.store_id = m.store_id
GROUP BY s.store_id, s.store_name, s.store_format, s.region
ORDER BY net_value_gbp DESC;
"""

result = pd.read_sql_query(query, conn)
result


Monthly


,store_id,store_name,store_format,region,member_revenue_gbp,redemption_cost_gbp,net_value_gbp,member_gp_after_cost_gbp,cost_pct_of_member_revenue
0,24,Sunrise Coffee Cheltenham Quarter,High-Street,South-West,966670.0,30060.0,936610.0,580689.0,3.11
1,50,Sunrise Coffee Bradford Brew Bar,High-Street,Yorkshire-and-Humber,915124.0,26656.0,888468.0,545828.0,2.91
2,21,Sunrise Coffee Birmingham Roastery,High-Street,West-Midlands,878854.0,30054.0,848800.0,506202.0,3.42
3,37,Sunrise Coffee Ipswich House,High-Street,East-of-England,874635.0,28327.0,846309.0,517734.0,3.24
4,52,Sunrise Coffee Chester House,High-Street,North-West,827062.0,29265.0,797797.0,489915.0,3.54
5,35,Sunrise Coffee Canterbury Yard,High-Street,South-East,821143.0,30296.0,790848.0,478684.0,3.69
6,57,Sunrise Coffee Luton House,Transport-Hub,East-of-England,809379.0,31922.0,777457.0,466252.0,3.94
7,17,Sunrise Coffee Sheffield Hub,High-Street,Yorkshire-and-Humber,797810.0,26458.0,771352.0,470960.0,3.32
8,56,Sunrise Coffee Lincoln Corner,High-Street,East-Midlands,775739.0,28681.0,747058.0,441563.0,3.70
9,11,Sunrise Coffee Cambridge Brew Bar,High-Street,East-of-England,769485.0,30086.0,739399.0,447210.0,3.91


In [15]:
# Does `loyalty_adoption_rate` vary significantly by store format or region?


print("Monthly")
query = """
SELECT
    'Format'                                                           AS breakdown,
    store_format                                                       AS grp,
    COUNT(*)                                                           AS stores,
    ROUND(100.0 * AVG(loyalty_adoption_rate), 1)                       AS avg_adoption_pct,
    ROUND(100.0 * MIN(loyalty_adoption_rate), 1)                       AS min_adoption_pct,
    ROUND(100.0 * MAX(loyalty_adoption_rate), 1)                       AS max_adoption_pct
FROM dim_store
GROUP BY store_format
UNION ALL
SELECT
    'Region',
    region,
    COUNT(*),
    ROUND(100.0 * AVG(loyalty_adoption_rate), 1),
    ROUND(100.0 * MIN(loyalty_adoption_rate), 1),
    ROUND(100.0 * MAX(loyalty_adoption_rate), 1)
FROM dim_store
GROUP BY region
ORDER BY breakdown, avg_adoption_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,breakdown,grp,stores,avg_adoption_pct,min_adoption_pct,max_adoption_pct
0,Format,Transport-Hub,2,45.8,39.7,51.8
1,Format,High-Street,58,32.8,5.7,58.5
2,Region,West-Midlands,3,44.6,34.2,54.0
3,Region,North-West,5,43.7,34.5,55.5
4,Region,South-West,5,41.6,25.9,58.5
5,Region,East-of-England,6,39.8,10.0,54.6
6,Region,North-East,4,36.4,31.0,48.0
7,Region,Yorkshire-and-Humber,6,34.5,10.8,57.3
8,Region,South-East,6,34.3,7.9,56.5
9,Region,Northern-Ireland,3,33.8,8.2,47.7


In [16]:
## Loyalty program based on high-street and transport-hub 


## Revenue Mix

In [17]:
# What proportion of daily revenue comes from espresso 
# drinks vs food vs filter coffee vs retail beans, 
# and how does this split differ by store format?


print("Monthly")
query = """
SELECT
    s.store_format,
    COUNT(*)                                                           AS day_records,
    ROUND(100.0 * SUM(d.espresso_revenue_gbp)     / SUM(d.total_revenue_gbp), 1) AS espresso_pct,
    ROUND(100.0 * SUM(d.filter_revenue_gbp)       / SUM(d.total_revenue_gbp), 1) AS filter_pct,
    ROUND(100.0 * SUM(d.food_revenue_gbp)         / SUM(d.total_revenue_gbp), 1) AS food_pct,
    ROUND(100.0 * SUM(d.retail_beans_revenue_gbp) / SUM(d.total_revenue_gbp), 1) AS retail_beans_pct
FROM fact_daily_sales AS d
JOIN dim_store AS s
    ON s.store_id = d.store_id
GROUP BY s.store_format
UNION ALL
SELECT
    'All stores',
    COUNT(*),
    ROUND(100.0 * SUM(espresso_revenue_gbp)     / SUM(total_revenue_gbp), 1),
    ROUND(100.0 * SUM(filter_revenue_gbp)       / SUM(total_revenue_gbp), 1),
    ROUND(100.0 * SUM(food_revenue_gbp)         / SUM(total_revenue_gbp), 1),
    ROUND(100.0 * SUM(retail_beans_revenue_gbp) / SUM(total_revenue_gbp), 1)
FROM fact_daily_sales;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,store_format,day_records,espresso_pct,filter_pct,food_pct,retail_beans_pct
0,High-Street,9686,45.7,19.1,26.1,9.1
1,Transport-Hub,314,46.1,20.4,24.9,8.6
2,All stores,10000,45.7,19.2,26.1,9.0


In [18]:
# Which product categories in dim_product carry the highest 
# gross_margin_pct, and are those products growing as a share of 
# daily revenue?


print("Monthly")
query = """
SELECT
    product_category,
    COUNT(*)                                                           AS products,
    ROUND(100.0 * AVG(gross_margin_pct), 1)                            AS avg_gross_margin_pct,
    ROUND(100.0 * MIN(gross_margin_pct), 1)                            AS min_gross_margin_pct,
    ROUND(100.0 * MAX(gross_margin_pct), 1)                            AS max_gross_margin_pct
FROM dim_product
GROUP BY product_category
ORDER BY avg_gross_margin_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result



Monthly


,product_category,products,avg_gross_margin_pct,min_gross_margin_pct,max_gross_margin_pct
0,Filter-Coffee,4,88.3,85.7,90.4
1,Espresso-Drinks,12,83.9,80.7,87.5
2,Cold-Drinks,4,81.3,77.1,85.0
3,Food,10,74.0,70.8,75.2
4,Merchandise,2,70.2,67.9,72.5
5,Retail-Beans,8,67.7,66.0,69.4


In [19]:
print("Monthly")
query = """
SELECT
    dt.year,
    dt.quarter,
    COUNT(*)                                                           AS day_records,
    ROUND(100.0 * SUM(d.espresso_revenue_gbp)     / SUM(d.total_revenue_gbp), 2) AS espresso_pct,
    ROUND(100.0 * SUM(d.filter_revenue_gbp)       / SUM(d.total_revenue_gbp), 2) AS filter_pct,
    ROUND(100.0 * SUM(d.food_revenue_gbp)         / SUM(d.total_revenue_gbp), 2) AS food_pct,
    ROUND(100.0 * SUM(d.retail_beans_revenue_gbp) / SUM(d.total_revenue_gbp), 2) AS retail_beans_pct
FROM fact_daily_sales AS d
JOIN dim_date AS dt
    ON dt.date_id = d.date_id
GROUP BY dt.year, dt.quarter
ORDER BY dt.year, dt.quarter;
"""

result = pd.read_sql_query(query, conn)
result


Monthly


,year,quarter,day_records,espresso_pct,filter_pct,food_pct,retail_beans_pct
0,2023,1,1264,45.89,19.00,26.14,8.98
1,2023,2,1285,45.75,18.68,26.93,8.65
2,2023,3,1240,45.91,19.34,25.77,8.99
3,2023,4,1248,45.78,19.12,26.12,8.98
4,2024,1,1242,44.61,19.66,26.46,9.27
5,2024,2,1259,46.54,19.00,25.67,8.80
6,2024,3,1260,45.65,19.19,25.77,9.38
7,2024,4,1202,45.41,19.53,25.71,9.35


In [20]:
# Is retail_beans_revenue_gbp growing as a proportion of 
# total revenue — suggesting a stronger at-home coffee consumer base?

print("Monthly")
query = """
WITH monthly_share AS (
    SELECT
        dt.year,
        dt.month_number,
        dt.month_name,
        100.0 * SUM(d.retail_beans_revenue_gbp) / SUM(d.total_revenue_gbp) AS beans_pct
    FROM fact_daily_sales AS d
    JOIN dim_date AS dt
        ON dt.date_id = d.date_id
    GROUP BY dt.year, dt.month_number, dt.month_name
)
SELECT
    year,
    month_number,
    month_name,
    ROUND(beans_pct, 2)                                                AS retail_beans_pct,
    ROUND(beans_pct - LAG(beans_pct) OVER (PARTITION BY month_number ORDER BY year), 2)
                                                                       AS change_vs_last_year_pts
FROM monthly_share
ORDER BY year, month_number;
"""

result = pd.read_sql_query(query, conn)
result


Monthly


,year,month_number,month_name,retail_beans_pct,change_vs_last_year_pts
0,2023,1,January,9.09,NaN
1,2023,2,February,8.85,NaN
2,2023,3,March,8.99,NaN
3,2023,4,April,9.62,NaN
4,2023,5,May,8.79,NaN
5,2023,6,June,7.66,NaN
6,2023,7,July,8.82,NaN
7,2023,8,August,9.16,NaN
8,2023,9,September,9.00,NaN
9,2023,10,October,9.01,NaN


## Seasonal and Temporal Patterns

In [21]:
# How does total_revenue_gbp differ between weekday and 
# weekend trading across formats?

print("Monthly")
query = """
SELECT
    s.store_format,
    SUM(CASE WHEN dt.is_weekend = 0 THEN 1 ELSE 0 END)                 AS weekday_records,
    SUM(CASE WHEN dt.is_weekend = 1 THEN 1 ELSE 0 END)                 AS weekend_records,
    ROUND(AVG(CASE WHEN dt.is_weekend = 0 THEN d.total_revenue_gbp END), 0) AS avg_weekday_revenue_gbp,
    ROUND(AVG(CASE WHEN dt.is_weekend = 1 THEN d.total_revenue_gbp END), 0) AS avg_weekend_revenue_gbp,
    ROUND(100.0 * (AVG(CASE WHEN dt.is_weekend = 1 THEN d.total_revenue_gbp END)
                 - AVG(CASE WHEN dt.is_weekend = 0 THEN d.total_revenue_gbp END))
                 / AVG(CASE WHEN dt.is_weekend = 0 THEN d.total_revenue_gbp END), 1)
                                                                       AS weekend_vs_weekday_pct
FROM fact_daily_sales AS d
JOIN dim_store AS s  ON s.store_id = d.store_id
JOIN dim_date  AS dt ON dt.date_id = d.date_id
GROUP BY s.store_format;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,store_format,weekday_records,weekend_records,avg_weekday_revenue_gbp,avg_weekend_revenue_gbp,weekend_vs_weekday_pct
0,High-Street,6948,2738,2178.0,1652.0,-24.2
1,Transport-Hub,240,74,2701.0,1983.0,-26.6


In [22]:
# Do bank holidays help or hurt revenue for Transport-Hub stores compared with High-Street stores?


print("Monthly")
query = """
SELECT
    s.store_format,
    SUM(CASE WHEN dt.is_uk_bank_holiday = 1 THEN 1 ELSE 0 END)         AS bank_holiday_records,
    ROUND(AVG(CASE WHEN dt.is_uk_bank_holiday = 0 AND dt.is_weekend = 0
                   THEN d.total_revenue_gbp END), 0)                   AS avg_normal_weekday_gbp,
    ROUND(AVG(CASE WHEN dt.is_uk_bank_holiday = 1
                   THEN d.total_revenue_gbp END), 0)                   AS avg_bank_holiday_gbp,
    ROUND(100.0 * (AVG(CASE WHEN dt.is_uk_bank_holiday = 1 THEN d.total_revenue_gbp END)
                 - AVG(CASE WHEN dt.is_uk_bank_holiday = 0 AND dt.is_weekend = 0
                            THEN d.total_revenue_gbp END))
                 / AVG(CASE WHEN dt.is_uk_bank_holiday = 0 AND dt.is_weekend = 0
                            THEN d.total_revenue_gbp END), 1)          AS bank_holiday_vs_weekday_pct
FROM fact_daily_sales AS d
JOIN dim_store AS s  ON s.store_id = d.store_id
JOIN dim_date  AS dt ON dt.date_id = d.date_id
GROUP BY s.store_format;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,store_format,bank_holiday_records,avg_normal_weekday_gbp,avg_bank_holiday_gbp,bank_holiday_vs_weekday_pct
0,High-Street,237,2207.0,1365.0,-38.1
1,Transport-Hub,8,2732.0,1794.0,-34.3


In [23]:
# Which months in 2023–2024 show the highest and lowest 
# net_profit_gbp across the estate, and does the pattern 
# repeat year-on-year?


print("Monthly")
query = """
WITH monthly AS (
    SELECT
        dt.year,
        dt.month_number,
        dt.month_name,
        SUM(m.net_profit_gbp)                                          AS net_profit
    FROM fact_store_monthly_profitability AS m
    JOIN dim_date AS dt
        ON dt.date_id = m.date_id
    GROUP BY dt.year, dt.month_number, dt.month_name
),
ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY year ORDER BY net_profit DESC)   AS rank_in_year
    FROM monthly
)
SELECT
    month_number,
    month_name,
    ROUND(MAX(CASE WHEN year = 2023 THEN net_profit END), 0)           AS net_profit_2023_gbp,
    MAX(CASE WHEN year = 2023 THEN rank_in_year END)                   AS rank_2023,
    ROUND(MAX(CASE WHEN year = 2024 THEN net_profit END), 0)           AS net_profit_2024_gbp,
    MAX(CASE WHEN year = 2024 THEN rank_in_year END)                   AS rank_2024
FROM ranked
GROUP BY month_number, month_name
ORDER BY month_number;
"""

result = pd.read_sql_query(query, conn)
result

Monthly


,month_number,month_name,net_profit_2023_gbp,rank_2023,net_profit_2024_gbp,rank_2024
0,1,January,707866.0,6,736028.0,5
1,2,February,679644.0,10,737533.0,4
2,3,March,742999.0,3,799319.0,2
3,4,April,761388.0,2,703490.0,7
4,5,May,689037.0,8,619730.0,11
5,6,June,700972.0,7,705257.0,6
6,7,July,807737.0,1,611204.0,12
7,8,August,739755.0,4,640444.0,9
8,9,September,660970.0,11,634822.0,10
9,10,October,687043.0,9,744435.0,3
